In [1]:

# CELL 1: Environment, adapter files, and Hugging Face token verification

import os
import torch
from pathlib import Path

print("=" * 60)
print("1. ENVIRONMENT CHECK")
print("=" * 60)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Check Kaggle Accelerator settings.")

print("\n" + "=" * 60)
print("2. ADAPTER FILE CHECK")
print("=" * 60)

DATASET_DIR = Path("/kaggle/input/datasets/ananyaaaa5/ai-fragment")

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "adapter_config_tone.json",
    "adapter_model (1).safetensors"
]

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATASET_DIR}\n"
        "Check whether ai-fragment is attached to this notebook."
    )

for filename in required_files:
    file_path = DATASET_DIR / filename

    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"FOUND: {filename} ({size_mb:.2f} MB)")
    else:
        print(f"MISSING: {filename}")

missing_files = [
    filename for filename in required_files
    if not (DATASET_DIR / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"\nMissing adapter files: {missing_files}"
    )

print("\nAll four adapter files are present.")

print("\n" + "=" * 60)
print("3. HUGGING FACE TOKEN CHECK")
print("=" * 60)

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi

    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")

    if not hf_token:
        raise ValueError("HF_TOKEN secret is empty.")

    # Validate the token without printing it
    api = HfApi()
    user_info = api.whoami(token=hf_token)

    print("HF_TOKEN: Available and valid")
    print("Authenticated Hugging Face account:", user_info.get("name", "Verified"))

except Exception as e:
    hf_token = None
    print("Hugging Face authentication failed.")
    print("Error:", str(e))
    raise RuntimeError(
        "Please check your Kaggle HF_TOKEN secret and its permissions."
    )

print("\n" + "=" * 60)
print("CELL 1 COMPLETED")
print("=" * 60)


1. ENVIRONMENT CHECK
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

2. ADAPTER FILE CHECK
FOUND: adapter_config.json (0.00 MB)
FOUND: adapter_model.safetensors (160.06 MB)
FOUND: adapter_config_tone.json (0.00 MB)
FOUND: adapter_model (1).safetensors (160.06 MB)

All four adapter files are present.

3. HUGGING FACE TOKEN CHECK
HF_TOKEN: Available and valid
Authenticated Hugging Face account: Ananya5

CELL 1 COMPLETED


In [4]:

%pip install -q --only-binary=:all: sentencepiece==0.2.3


Note: you may need to restart the kernel to use updated packages.


In [5]:

import sentencepiece
import torch

print("SentencePiece:", sentencepiece.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


SentencePiece: 0.2.3
PyTorch: 2.11.0+cu130
CUDA available: True


In [6]:
import importlib.metadata as metadata
import torch

packages = [
    "transformers",
    "peft",
    "accelerate",
    "huggingface-hub",
    "safetensors",
    "datasets",
    "sentencepiece",
    "bitsandbytes",
    "rouge-score",
    "jsonschema",
]

print("=== ENVIRONMENT CHECK ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n=== PACKAGE VERSIONS ===")
for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nEnvironment check complete.")

=== ENVIRONMENT CHECK ===
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

=== PACKAGE VERSIONS ===
transformers: 5.20.2
peft: 0.21.2
accelerate: 1.15.0
huggingface-hub: 1.33.0
safetensors: 0.8.0
datasets: 4.8.5
sentencepiece: 0.2.3
bitsandbytes: NOT INSTALLED
rouge-score: 0.1.2
jsonschema: 4.26.0

Environment check complete.


In [7]:
%pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 129.9 MB/s  0:00:00m0:00:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
import bitsandbytes as bnb
import transformers, peft, torch

print("bitsandbytes:", bnb.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("\nDEPENDENCY CHECK PASSED")

bitsandbytes: 0.50.2
Transformers: 5.20.2
PEFT: 0.21.2
PyTorch: 2.11.0+cu130
GPU: Tesla T4

DEPENDENCY CHECK PASSED


In [9]:

import json
import shutil
import hashlib
from pathlib import Path
from huggingface_hub import snapshot_download

# --------------------------------------------------
# 1. Paths and adapter details
# --------------------------------------------------

SOURCE = Path("/kaggle/input/datasets/ananyaaaa5/ai-fragment")
OUTPUT = Path("/kaggle/working/adapters")
OUTPUT.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

ADAPTERS = {
    "json": {
        "source_config": SOURCE / "adapter_config.json",
        "source_weights": SOURCE / "adapter_model.safetensors",
        "sha256": "a457bc8b686518e5a39bbcb8975eb478c7c7f1f6b891f45a74527db69b6532fb",
    },
    "tone": {
        "source_config": SOURCE / "adapter_config_tone.json",
        "source_weights": SOURCE / "adapter_model (1).safetensors",
        "sha256": "705ef1266e1351d37a303877d065495591be5eb1ef07aa583d1f0c4fc53db778",
    },
    "sql": {
        "repo": "Lots-of-LoRAs/Mistral-7B-Instruct-v0.2-4b-r16-task107",
        "revision": "3c4df4d2046d03efd4cab9c08968434e80ef5e4b",
        "sha256": "8e1179f274449a1f0acae1e54bf92c7c7ce91cc69ea12c6fc1aed20740f5ec22",
    },
    "samsum": {
        "repo": "Lots-of-LoRAs/Mistral-7B-Instruct-v0.2-4b-r16-task1572",
        "revision": "91dac07507db426fe186cf23049ad219e4a420ba",
        "sha256": "d115681fa8f051f840569f6cdd688064a35b709c2a72e4a59a404f293c8fad3d",
    },
}

# --------------------------------------------------
# 2. Copy in-house adapters
# --------------------------------------------------

for name in ["json", "tone"]:
    info = ADAPTERS[name]
    folder = OUTPUT / name
    folder.mkdir(parents=True, exist_ok=True)

    for key in ["source_config", "source_weights"]:
        if not info[key].exists():
            raise FileNotFoundError(f"Missing file: {info[key]}")

    shutil.copy2(info["source_config"], folder / "adapter_config.json")
    shutil.copy2(info["source_weights"], folder / "adapter_model.safetensors")

# --------------------------------------------------
# 3. Download external adapters at pinned commits
# --------------------------------------------------

for name in ["sql", "samsum"]:
    info = ADAPTERS[name]
    folder = OUTPUT / name

    snapshot_download(
        repo_id=info["repo"],
        revision=info["revision"],
        local_dir=str(folder),
        allow_patterns=["adapter_config.json", "adapter_model.safetensors"],
    )

# --------------------------------------------------
# 4. Verify hashes and configuration
# --------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = {}

for name, info in ADAPTERS.items():
    folder = OUTPUT / name
    config_path = folder / "adapter_config.json"
    weights_path = folder / "adapter_model.safetensors"

    if not config_path.exists() or not weights_path.exists():
        raise FileNotFoundError(f"Adapter files missing for {name}")

    actual_hash = sha256_file(weights_path)

    if actual_hash != info["sha256"]:
        raise ValueError(f"SHA-256 mismatch for {name}")

    with open(config_path, "r") as file:
        config = json.load(file)

    base = config.get("base_model_name_or_path")
    rank = config.get("r")
    alpha = config.get("lora_alpha")

    if base != BASE_MODEL:
        raise ValueError(f"Unexpected base model for {name}: {base}")

    if rank != 16 or alpha != 32:
        raise ValueError(
            f"Unexpected LoRA settings for {name}: rank={rank}, alpha={alpha}"
        )

    manifest[name] = {
        "base_model": base,
        "rank": rank,
        "alpha": alpha,
        "target_modules": config.get("target_modules"),
        "sha256": actual_hash,
        "repo": info.get("repo", "in-house"),
        "revision": info.get("revision", config.get("revision")),
    }

    print(f"{name.upper()}: PASS")
    print(f"  Rank: {rank} | Alpha: {alpha}")
    print(f"  SHA-256: MATCH")

# --------------------------------------------------
# 5. Save manifest
# --------------------------------------------------

manifest_path = OUTPUT / "adapters_manifest.json"

with open(manifest_path, "w") as file:
    json.dump(manifest, file, indent=4)

print("\nALL FOUR ADAPTERS VERIFIED.")
print("Manifest saved:", manifest_path)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

JSON: PASS
  Rank: 16 | Alpha: 32
  SHA-256: MATCH
TONE: PASS
  Rank: 16 | Alpha: 32
  SHA-256: MATCH
SQL: PASS
  Rank: 16 | Alpha: 32
  SHA-256: MATCH
SAMSUM: PASS
  Rank: 16 | Alpha: 32
  SHA-256: MATCH

ALL FOUR ADAPTERS VERIFIED.
Manifest saved: /kaggle/working/adapters/adapters_manifest.json


In [10]:

import json
import torch
from pathlib import Path
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Model details
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

# Resolve and pin the exact model revision
BASE_REVISION = HfApi().model_info(BASE_MODEL).sha

print("Base model:", BASE_MODEL)
print("Pinned revision:", BASE_REVISION)

# 2. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    revision=BASE_REVISION
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# 3. Configure 4-bit quantization for Tesla T4
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 4. Load frozen base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=BASE_REVISION,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16
)

base_model.eval()
base_model.config.use_cache = True

# 5. Record the pinned revision
config_path = Path("/kaggle/working/adapters/phase1_config.json")

phase_config = {
    "base_model": BASE_MODEL,
    "base_revision": BASE_REVISION,
    "quantization": "4-bit NF4",
    "compute_dtype": "float16"
}

with open(config_path, "w") as f:
    json.dump(phase_config, f, indent=4)

print("\nBASE MODEL LOADED SUCCESSFULLY")
print("Model device:", base_model.device)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Base model remains frozen:", not any(
    p.requires_grad for p in base_model.parameters()
))


Base model: mistralai/Mistral-7B-Instruct-v0.2
Pinned revision: 63a8b081895390a26e140280378bc85ec8bce07a


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


BASE MODEL LOADED SUCCESSFULLY
Model device: cuda:0
GPU memory allocated: 1.71 GB
Base model remains frozen: False


In [12]:
for name, obj in globals().copy().items():
    if hasattr(obj, "parameters") and hasattr(obj, "config"):
        print("Model variable found:", name)
        print("Model type:", type(obj).__name__)

Model variable found: base_model
Model type: MistralForCausalLM


In [13]:
# Freeze the base model
for param in base_model.parameters():
    param.requires_grad = False

# Verify
trainable_params = sum(
    p.numel() for p in base_model.parameters() if p.requires_grad
)

total_params = sum(p.numel() for p in base_model.parameters())

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Base model frozen:", trainable_params == 0)

assert trainable_params == 0

print("\nBASE MODEL FREEZING VERIFIED")

Total parameters: 3,752,071,168
Trainable parameters: 0
Base model frozen: True

BASE MODEL FREEZING VERIFIED


In [14]:
from peft import PeftModel

# Load JSON adapter onto the frozen base model
model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/adapters/json",
    adapter_name="json",
    is_trainable=False
)

# Verify adapter
print("Loaded adapter:", model.active_adapter)
print("Base model:", model.base_model.config._name_or_path)

# Verify trainable parameters
trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Trainable parameters:", f"{trainable:,}")
print("\nJSON ADAPTER LOADED SUCCESSFULLY")

Loaded adapter: json
Base model: mistralai/Mistral-7B-Instruct-v0.2
Trainable parameters: 0

JSON ADAPTER LOADED SUCCESSFULLY


In [15]:
# Load the remaining adapters
for name in ["tone", "sql", "samsum"]:
    model.load_adapter(
        f"/kaggle/working/adapters/{name}",
        adapter_name=name,
        is_trainable=False
    )
    print(f"{name.upper()} adapter loaded successfully")

# Verify all adapters
print("\nAvailable adapters:", list(model.peft_config.keys()))
print("Active adapter:", model.active_adapter)

# Verify base and adapters are frozen
trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Trainable parameters:", f"{trainable:,}")

assert set(model.peft_config.keys()) == {"json", "tone", "sql", "samsum"}
assert trainable == 0

print("\nALL FOUR ADAPTERS LOADED AND FROZEN")

TONE adapter loaded successfully
SQL adapter loaded successfully
SAMSUM adapter loaded successfully

Available adapters: ['json', 'tone', 'sql', 'samsum']
Active adapter: json
Trainable parameters: 0

ALL FOUR ADAPTERS LOADED AND FROZEN


In [16]:

import torch

tokenizer = tokenizer if "tokenizer" in globals() else None

if tokenizer is None:
    raise NameError("Tokenizer not found. Load the tokenizer before running this cell.")

test_prompts = {
    "json": """Extract the information from the text as JSON that follows the schema.
Output JSON only, no explanation.

Instruction: Extract the person's name and age.
Schema: {"name": "string", "age": "integer"}
Text: Rahul is 24 years old.""",

    "tone": """Write a polite and empathetic customer support reply.

Customer: My order has not arrived yet, and I am very disappointed.""",

    "sql": """Convert the question into SQL.

Question: Show all employees whose salary is greater than 50000.
Table: employees(id, name, salary)""",

    "samsum": """Summarize this conversation briefly.

Amanda: Are we still meeting tomorrow?
John: Yes, at 10 AM.
Amanda: Great, see you then!"""
}

model.eval()

for adapter_name, prompt in test_prompts.items():
    model.set_adapter(adapter_name)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    print(f"\n{'='*55}")
    print(f"ADAPTER: {adapter_name.upper()}")
    print("="*55)
    print(response.strip())

print("\nFOUR-ADAPTER GENERATION TEST COMPLETED")



ADAPTER: JSON


ADAPTER: TONE
Support: We apologize for the inconvenience. Could you please provide your order number so we can look into the status of your delivery? We will do our best to assist you and provide a solution as soon as possible.

ADAPTER: SQL
SELECT * FROM employees WHERE salary > 50000

ADAPTER: SAMSUM
John: Sounds good.
 Amanda: I'll bring the documents.
 John: I'll bring the laptop.
 Amanda: Perfect.
 John: I'll send you the location details later.
 Amanda: Okay, I'll text you when I'm on my way.
 John: Sounds good.
 Amanda: See you tomorrow!
 John: See you tomorrow!

FOUR-ADAPTER GENERATION TEST COMPLETED


In [17]:
model.set_adapter("json")

prompt = """Extract the information from the text as JSON that follows the schema.
Output JSON only, no explanation.

Instruction: Extract the person's name and age.
Schema: {"name": "string", "age": "integer"}
Text: Rahul is 24 years old."""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

print("Input tokens:", inputs["input_ids"].shape[1])

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

print("Generated token count:", output.shape[1] - inputs["input_ids"].shape[1])
print("Raw decoded output:")
print(repr(tokenizer.decode(output[0], skip_special_tokens=False)))

Input tokens: 64
Generated token count: 1
Raw decoded output:
'<s> Extract the information from the text as JSON that follows the schema.\nOutput JSON only, no explanation.\n\nInstruction: Extract the person\'s name and age.\nSchema: {"name": "string", "age": "integer"}\nText: Rahul is 24 years old.</s>'


In [18]:
import torch

prompt = """Extract the information from the text as JSON that follows the schema.
Output JSON only, no explanation.

Instruction: Extract the person's name and age.
Schema: {"name": "string", "age": "integer"}
Text: Rahul is 24 years old."""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

for name in ["base", "json"]:
    if name == "base":
        model.disable_adapter()
    else:
        model.enable_adapter_layers()
        model.set_adapter("json")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]

    print(f"\n{name.upper()}")
    print("Token IDs:", new_tokens.tolist())
    print("Output:", repr(tokenizer.decode(new_tokens, skip_special_tokens=False)))

# Restore JSON adapter
model.enable_adapter_layers()
model.set_adapter("json")


BASE
Token IDs: [2]
Output: '</s>'

JSON
Token IDs: [2]
Output: '</s>'


In [19]:
import torch

prompt = """Extract the information from the text as JSON that follows the schema.
Output JSON only, no explanation.

Instruction: Extract the person's name and age.
Schema: {"name": "string", "age": "integer"}
Text: Rahul is 24 years old."""

# Tokenize without automatically appending EOS
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False
)

# Add BOS manually if required
if tokenizer.bos_token_id is not None:
    bos = torch.tensor(
        [[tokenizer.bos_token_id]],
        device=inputs["input_ids"].device
    )
    inputs["input_ids"] = torch.cat(
        [bos, inputs["input_ids"]], dim=1
    )
    inputs["attention_mask"] = torch.ones_like(inputs["input_ids"])

inputs = inputs.to(model.device)

print("Last input token:", inputs["input_ids"][0, -1].item())
print("EOS token ID:", tokenizer.eos_token_id)

for name in ["base", "json"]:
    if name == "base":
        model.disable_adapter()
    else:
        model.enable_adapter_layers()
        model.set_adapter("json")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0, inputs["input_ids"].shape[1]:]

    print(f"\n{name.upper()} OUTPUT:")
    print(tokenizer.decode(generated, skip_special_tokens=True))

model.enable_adapter_layers()
model.set_adapter("json")

Last input token: 28723
EOS token ID: 2

BASE OUTPUT:


JSON OUTPUT:



In [20]:
import torch

model.eval()
model.set_adapter("json")

with torch.inference_mode():
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

# Get the most likely next tokens
logits = outputs.logits[0, -1, :]
top_values, top_ids = torch.topk(logits, 10)

print("Top 10 predicted next tokens:\n")

for token_id, score in zip(top_ids.tolist(), top_values.tolist()):
    print(
        f"Token ID: {token_id:6d} | "
        f"Token: {repr(tokenizer.decode([token_id])):20s} | "
        f"Score: {score:.4f}"
    )

print("\nEOS score:", outputs.logits[0, -1, tokenizer.eos_token_id].item())

Top 10 predicted next tokens:

Token ID:      2 | Token: '</s>'               | Score: 15.8594
Token ID:     13 | Token: '\n'                 | Score: 15.8281
Token ID:    650 | Token: 'He'                 | Score: 13.8281
Token ID:   2354 | Token: 'His'                | Score: 11.7188
Token ID:   2215 | Token: 'John'               | Score: 10.8750
Token ID:    415 | Token: 'The'                | Score: 10.6328
Token ID:   6620 | Token: 'Name'               | Score: 9.5938
Token ID:   6441 | Token: 'Person'             | Score: 9.1953
Token ID:    851 | Token: 'This'               | Score: 8.5859
Token ID:    414 | Token: '\\'                 | Score: 8.1719

EOS score: 15.859375


In [22]:
import torch

prompt = """Extract the information from the text as JSON that follows the schema.
Output JSON only, no explanation.

Instruction: Extract the person's name and age.
Schema: {"name": "string", "age": "integer"}
Text: Rahul is 24 years old."""

messages = [{"role": "user", "content": prompt}]

chat_inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# Extract input IDs and attention mask
input_ids = chat_inputs["input_ids"]
attention_mask = chat_inputs["attention_mask"]

print("Input shape:", input_ids.shape)

for name in ["base", "json"]:
    if name == "base":
        model.disable_adapter()
    else:
        model.enable_adapter_layers()
        model.set_adapter("json")

    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0, input_ids.shape[1]:]

    print(f"\n{name.upper()} OUTPUT:")
    print(repr(tokenizer.decode(generated, skip_special_tokens=True)))

model.enable_adapter_layers()
model.set_adapter("json")

Input shape: torch.Size([1, 71])

BASE OUTPUT:
'{"name": "Rahul", "age": 24}'

JSON OUTPUT:
'{"name": "Rahul", "age": 24}'


In [23]:

import torch

test_prompts = {
    "json": """Extract the information as JSON.
Instruction: Extract name and age.
Schema: {"name": "string", "age": "integer"}
Text: Rahul is 24 years old.""",

    "tone": """Write a polite and empathetic customer support reply.
Customer: My order has not arrived yet, and I am disappointed.""",

    "sql": """Convert the question into SQL.
Question: Show all employees whose salary is greater than 50000.
Table: employees(id, name, salary)""",

    "samsum": """Summarize this conversation briefly.
Amanda: Are we still meeting tomorrow?
John: Yes, at 10 AM.
Amanda: Great, see you then!"""
}

model.eval()

for name, prompt in test_prompts.items():
    messages = [{"role": "user", "content": prompt}]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    model.set_adapter(name)

    with torch.inference_mode():
        output = model.generate(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"],
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0, encoded["input_ids"].shape[1]:]

    print(f"\n{'='*45}")
    print(f"{name.upper()} ADAPTER")
    print("="*45)
    print(tokenizer.decode(generated, skip_special_tokens=True))

print("\nALL FOUR CHAT-FORMAT TESTS COMPLETED")



JSON ADAPTER
{"name": "Rahul", "age": 24}

TONE ADAPTER
We apologize for the inconvenience. Could you please provide your order number so we can look into the status of your delivery? We will work to resolve the issue and get your order to you as soon as possible.

SQL ADAPTER
SELECT id, name FROM employees WHERE salary > 50000

SAMSUM ADAPTER
Amanda and John confirm their meeting tomorrow at 10 AM. Amanda expresses her excitement about the upcoming meeting.

ALL FOUR CHAT-FORMAT TESTS COMPLETED


In [29]:
from datasets import load_dataset

evaluation_sources = {
    "json": "paraloq/json_data_extraction",
    "tone": "Tobi-Bueck/customer-support-tickets",
    "sql": "Lots-of-LoRAs/task107_splash_question_to_sql",
    "samsum": "Samsung/samsum"
}

eval_datasets = {}

for task, dataset_name in evaluation_sources.items():
    print(f"\n{'='*55}")
    print(f"Loading: {task} | {dataset_name}")

    try:
        dataset = load_dataset(dataset_name)
        eval_datasets[task] = dataset

        print("Available splits:", list(dataset.keys()))

        for split_name, split_data in dataset.items():
            print(f"\n{split_name}: {len(split_data)} examples")
            print("Columns:", split_data.column_names)
            print("Sample:", split_data[0])

    except Exception as e:
        print(f"Could not load {task}: {e}")


Loading: json | paraloq/json_data_extraction


README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data.parquet: reconstructing file:   0%|          |  0.00B /  967kB            

data.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/484 [00:00<?, ? examples/s]

Available splits: ['train']

train: 484 examples
Columns: ['title', 'topic', 'item_id', 'schema', 'item', 'text', 'medium', '__index_level_0__']
Sample: {'title': 'Prescription Medication', 'topic': 'medical', 'item_id': 'PrescriptionMedication', 'schema': '{"$schema": "http://json-schema.org/draft-07/schema#", "$defs": {"Patient": {"type": "object", "properties": {"patientId": {"type": "string", "minLength": 1, "maxLength": 255}, "firstName": {"type": "string", "minLength": 1, "maxLength": 255}, "lastName": {"type": "string", "minLength": 1, "maxLength": 255}, "dateOfBirth": {"type": "string", "format": "date"}, "gender": {"type": "string", "enum": ["male", "female", "other"]}, "medicalConditions": {"type": "array", "items": {"type": "string", "minLength": 1, "maxLength": 255}, "minItems": 0, "uniqueItems": true}}, "required": ["patientId", "firstName", "lastName", "dateOfBirth", "gender"]}, "Medication": {"type": "object", "properties": {"medicationId": {"type": "string", "minLength"

... (15 outputs left out)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating valid split: 0 examples [00:00, ? examples/s]

Available splits: ['train', 'test', 'valid']

train: 1624 examples
Columns: ['input', 'output', 'id']
Sample: {'input': 'Definition: In this task you are expected to write an SQL query that will return the data asked for in the question. An SQL query works by selecting data from a table where certain conditions apply. A table contains columns where every row in that table must have a value for each column. Every table has a primary key that uniquely identifies each row, usually an id. To choose which columns are returned you specify that after the "SELECT" statement. Next, you use a "FROM" statement to specify what tables you want to select the data from. When you specify a table you can rename it with the "AS" statement. You can reference that table by whatever name follows the "AS" statement. If you want to select data from multiple tables you need to use the "JOIN" statement. This will join the tables together by pairing a row in one table with every row in the other table (Cartesia

In [30]:
from datasets import load_dataset

eval_datasets["samsum"] = load_dataset("knkarthick/samsum")

print("SAMSum splits:", list(eval_datasets["samsum"].keys()))

for split, data in eval_datasets["samsum"].items():
    print(f"{split}: {len(data)} examples | Columns: {data.column_names}")

print("\nTest sample:")
print(eval_datasets["samsum"]["test"][0])

README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

SAMSum splits: ['train', 'validation', 'test']
train: 14731 examples | Columns: ['id', 'dialogue', 'summary']
validation: 818 examples | Columns: ['id', 'dialogue', 'summary']
test: 819 examples | Columns: ['id', 'dialogue', 'summary']

Test sample:
{'id': '13862856', 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye", 'summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry."}


In [31]:
from datasets import Dataset
import random
import json
import os

SEED = 42
random.seed(SEED)

# 1. JSON: fixed sample of 71
json_data = eval_datasets["json"]["train"].shuffle(seed=SEED)
json_eval = json_data.select(range(71))

# 2. Tone: English-language tickets only
tone_data = eval_datasets["tone"]["train"].filter(
    lambda x: x["language"] == "en"
).shuffle(seed=SEED)

tone_eval = tone_data.select(range(150))

# 3. SQL: official test split
sql_eval = eval_datasets["sql"]["test"]

# 4. SAMSum: fixed sample from official test split
samsum_data = eval_datasets["samsum"]["test"].shuffle(seed=SEED)
samsum_eval = samsum_data.select(range(150))

# Store all evaluation sets
frozen_eval = {
    "json": json_eval,
    "tone": tone_eval,
    "sql": sql_eval,
    "samsum": samsum_eval
}

# Save locally in Kaggle
os.makedirs("/kaggle/working/evaluation_sets", exist_ok=True)

for task, dataset in frozen_eval.items():
    dataset.to_json(
        f"/kaggle/working/evaluation_sets/{task}_eval.jsonl"
    )
    print(f"{task}: {len(dataset)} examples")

print("\nEvaluation sets saved successfully.")

Filter:   0%|          | 0/61765 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

json: 71 examples


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

tone: 150 examples


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

sql: 204 examples


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

samsum: 150 examples

Evaluation sets saved successfully.


In [32]:
# JSON
x = json_eval[0]
print("========== JSON ==========")
print("INPUT:")
print(f"Instruction: Extract all the information that the schema asks for.")
print(f"Schema: {x['schema'][:300]}...")
print(f"Text: {x['text'][:300]}...")
print("\nEXPECTED OUTPUT:")
print(x["item"][:300])

# TONE
x = tone_eval[0]
print("\n========== TONE ==========")
print("INPUT:")
print("Subject:", x["subject"])
print("Ticket:", x["body"][:300])
print("\nEXPECTED REPLY:")
print(x["answer"][:300])

# SQL
x = sql_eval[0]
print("\n========== SQL ==========")
print("INPUT:")
print(x["input"][-500:])
print("\nEXPECTED SQL:")
print(x["output"])

# SAMSUM
x = samsum_eval[0]
print("\n========== SAMSUM ==========")
print("INPUT:")
print(x["dialogue"][:500])
print("\nEXPECTED SUMMARY:")
print(x["summary"])

========== JSON ==========
INPUT:
Instruction: Extract all the information that the schema asks for.
Schema: {"$schema": "http://json-schema.org/draft-07/schema#", "$defs": {"Party": {"type": "object", "properties": {"partyID": {"type": "number"}, "name": {"type": "string"}, "email": {"type": "string", "format": "email"}, "phone": {"type": "string", "pattern": "^\\d{3}-\\d{3}-\\d{4}$"}, "address": {"type":...
Text: ## Order Details

**Order ID:** 1001

**Customer Information:**

* **Party ID:** 1
* **Name:** Acme Corporation
* **Email:** acme@example.com
* **Phone:** 555-123-4567
* **Address:** 123 Main Street
* **City:** Anytown
* **State:** CA
* **Zip:** 91234

**Product Information:**

* **Product ID:** 200...

EXPECTED OUTPUT:
{"orderID": 1001, "customer": {"partyID": 1, "name": "Acme Corporation", "email": "acme@example.com", "phone": "555-123-4567", "address": "123 Main Street", "city": "Anytown", "state": "CA", "zip": "91234"}, "product": {"productID": 2001, "name": "Widget", "d

In [34]:
import torch

def build_prompt(task, example):

    if task == "json":
        user_prompt = f"""Extract the information from the text as JSON that follows the schema.
Output JSON only, no explanation.

Instruction: Extract all the information that the schema asks for.

Schema:
{example['schema']}

Text:
{example['text']}"""

    elif task == "tone":
        user_prompt = f"""Write a helpful, empathetic and professional customer support reply.

Subject: {example['subject']}

Customer message:
{example['body']}

Write only the reply."""

    elif task == "sql":
        # Preserve the original Task 107 prompt format.
        user_prompt = example["input"]

    elif task == "samsum":
        user_prompt = f"""Summarize the following conversation briefly.

Conversation:
{example['dialogue']}

Summary:"""

    messages = [
        {"role": "user", "content": user_prompt}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def generate_response(task, example, max_new_tokens=400):

    prompt = build_prompt(task, example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(next(model.parameters()).device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


print("Common generation function ready.")

Common generation function ready.


In [35]:
import os
import json
import torch
from tqdm.auto import tqdm

# 1. Activate the original base model (no LoRA adapters)
model.disable_adapter_layers()
model.eval()

# 2. Reference-answer extraction
def get_reference(task, example):
    if task == "json":
        return example["item"]
    elif task == "tone":
        return example["answer"]
    elif task == "sql":
        output = example["output"]
        return output[0] if isinstance(output, list) else output
    elif task == "samsum":
        return example["summary"]

# 3. Evaluation settings
MAX_NEW_TOKENS = {
    "json": 400,
    "tone": 250,
    "sql": 200,
    "samsum": 150
}

base_results = {}
os.makedirs("/kaggle/working/results/base", exist_ok=True)

# 4. Generate predictions for every task
for task, dataset in frozen_eval.items():

    print(f"\n{'='*55}")
    print(f"BASE MODEL EVALUATION: {task.upper()}")
    print(f"Examples: {len(dataset)}")

    task_results = []

    for i, example in enumerate(tqdm(dataset, desc=task)):

        try:
            prediction = generate_response(
                task,
                example,
                max_new_tokens=MAX_NEW_TOKENS[task]
            )

            task_results.append({
                "example_id": str(example.get("id", i)),
                "prediction": prediction,
                "reference": get_reference(task, example)
            })

        except Exception as e:
            print(f"\nError on {task} example {i}: {e}")

            task_results.append({
                "example_id": str(example.get("id", i)),
                "prediction": "",
                "reference": get_reference(task, example),
                "error": str(e)
            })

    base_results[task] = task_results

    output_path = f"/kaggle/working/results/base/{task}_base.jsonl"

    with open(output_path, "w", encoding="utf-8") as f:
        for result in task_results:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

    successful = sum(1 for r in task_results if "error" not in r)

    print(f"Completed: {successful}/{len(dataset)}")
    print(f"Saved: {output_path}")

# 5. Restore adapters for subsequent Solo evaluation
model.enable_adapter_layers()

print("\n" + "="*55)
print("BASE EVALUATION COMPLETED")
print("Total examples:", sum(len(v) for v in base_results.values()))
print("Results directory: /kaggle/working/results/base/")


BASE MODEL EVALUATION: JSON
Examples: 71


json:   0%|          | 0/71 [00:00<?, ?it/s]


Error on json example 56: CUDA out of memory. Tried to allocate 9.01 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.77 GiB is free. Including non-PyTorch memory, this process has 12.79 GiB memory in use. Of the allocated memory 12.02 GiB is allocated by PyTorch, and 654.68 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
Completed: 70/71
Saved: /kaggle/working/results/base/json_base.jsonl

BASE MODEL EVALUATION: TONE
Examples: 150


tone:   0%|          | 0/150 [00:00<?, ?it/s]

Completed: 150/150
Saved: /kaggle/working/results/base/tone_base.jsonl

BASE MODEL EVALUATION: SQL
Examples: 204


sql:   0%|          | 0/204 [00:00<?, ?it/s]

Completed: 204/204
Saved: /kaggle/working/results/base/sql_base.jsonl

BASE MODEL EVALUATION: SAMSUM
Examples: 150


samsum:   0%|          | 0/150 [00:00<?, ?it/s]

Completed: 150/150
Saved: /kaggle/working/results/base/samsum_base.jsonl

BASE EVALUATION COMPLETED
Total examples: 575
Results directory: /kaggle/working/results/base/


In [36]:
import torch

# Clear unused GPU memory
torch.cuda.empty_cache()

# Inspect the failed JSON example
example = json_eval[56]
prompt = build_prompt("json", example)

tokens = tokenizer(prompt, return_tensors="pt")

print("JSON example index: 56")
print("Input tokens:", tokens["input_ids"].shape[1])
print("Schema characters:", len(example["schema"]))
print("Text characters:", len(example["text"]))
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("GPU memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

JSON example index: 56
Input tokens: 8692
Schema characters: 1367
Text characters: 127076
GPU memory allocated: 1.9 GB
GPU memory reserved: 2.01 GB


In [37]:
print("Attention implementation:",
      getattr(model.config, "_attn_implementation", "unknown"))

print("Model dtype:",
      next(model.parameters()).dtype)

print("Maximum context length:",
      model.config.max_position_embeddings)

print("GPU:", torch.cuda.get_device_name(0))

Attention implementation: sdpa
Model dtype: torch.float16
Maximum context length: 32768
GPU: Tesla T4


In [38]:
import gc
import torch
import json

gc.collect()
torch.cuda.empty_cache()

example = json_eval[56]

try:
    prediction = generate_response(
        "json",
        example,
        max_new_tokens=256
    )

    base_results["json"][56] = {
        "example_id": str(example.get("id", 56)),
        "prediction": prediction,
        "reference": get_reference("json", example)
    }

    # Update the saved JSON results
    with open(
        "/kaggle/working/results/base/json_base.jsonl",
        "w",
        encoding="utf-8"
    ) as f:
        for result in base_results["json"]:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

    print("Example 56 successfully evaluated.")
    print("\nPrediction:\n", prediction[:1000])

except torch.cuda.OutOfMemoryError:
    print("Still out of memory. Do not rerun the full evaluation.")
    gc.collect()
    torch.cuda.empty_cache()

Still out of memory. Do not rerun the full evaluation.


In [39]:
import json
from pathlib import Path

print("BASE EVALUATION SUMMARY")
print("=" * 45)

for task, results in base_results.items():
    successful = sum(
        1 for r in results
        if "error" not in r and r["prediction"].strip()
    )

    print(f"{task.upper():8} | {successful}/{len(results)} successful")

# Save a summary
summary = {
    task: {
        "total": len(results),
        "successful": sum(
            1 for r in results
            if "error" not in r and r["prediction"].strip()
        )
    }
    for task, results in base_results.items()
}

with open(
    "/kaggle/working/results/base/base_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=4)

print("\nSummary saved.")

BASE EVALUATION SUMMARY
JSON     | 70/71 successful
TONE     | 150/150 successful
SQL      | 204/204 successful
SAMSUM   | 150/150 successful

Summary saved.


In [40]:
import os
import json
import shutil
import hashlib
import zipfile
import importlib.metadata
from pathlib import Path
from datasets import load_dataset

ROOT = Path("/kaggle/working/experiment_checkpoint")
ROOT.mkdir(parents=True, exist_ok=True)

# 1. Save exact evaluation datasets
dataset_dir = ROOT / "evaluation_datasets"
dataset_dir.mkdir(exist_ok=True)

for task, dataset in frozen_eval.items():
    dataset.save_to_disk(str(dataset_dir / task))

# 2. Save all Base predictions, including recorded failures
base_dir = ROOT / "base_results"
base_dir.mkdir(exist_ok=True)

for task, results in base_results.items():
    with open(base_dir / f"{task}_base.jsonl", "w", encoding="utf-8") as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

# 3. Save experiment metadata
packages = [
    "torch", "transformers", "peft", "accelerate",
    "datasets", "huggingface-hub", "safetensors"
]

environment = {}
for package in packages:
    try:
        environment[package] = importlib.metadata.version(package)
    except Exception:
        environment[package] = "unknown"

metadata = {
    "project": "AI Model Fragment Compiler",
    "phase": "Phase 1 - Base evaluation",
    "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
    "base_revision": "63a8b081895390a26e140280378bc85ec8bce07a",
    "seed": 42,
    "generation": {
        "do_sample": False,
        "json_max_new_tokens": 400,
        "tone_max_new_tokens": 250,
        "sql_max_new_tokens": 200,
        "samsum_max_new_tokens": 150
    },
    "evaluation_sources": {
        "json": "paraloq/json_data_extraction",
        "tone": "Tobi-Bueck/customer-support-tickets (PROVISIONAL; possible training overlap)",
        "sql": "Lots-of-LoRAs/task107_splash_question_to_sql - test split",
        "samsum": "knkarthick/samsum - test split"
    },
    "base_success_counts": {
        task: sum(
            1 for r in results
            if "error" not in r and r["prediction"].strip()
        )
        for task, results in base_results.items()
    },
    "known_issue": "JSON example 56 failed with CUDA OOM",
    "environment": environment
}

with open(ROOT / "experiment_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

# 4. Copy adapter provenance manifest
manifest_path = Path("/kaggle/working/adapters/adapters_manifest.json")
if manifest_path.exists():
    shutil.copy2(manifest_path, ROOT / "adapters_manifest.json")

# 5. Load independent Tone evaluation source
print("Loading independent Tone dataset...")
tone_independent = load_dataset(
    "arpieb/consumer-electronics-support",
    split="train"
)

print("\nDataset rows:", len(tone_independent))
print("Columns:", tone_independent.column_names)
print("\nFirst example:")
print(tone_independent[0])

# 6. Create a checkpoint archive
archive_path = "/kaggle/working/phase1_base_checkpoint.zip"

with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in ROOT.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(ROOT.parent))

print("\nCHECKPOINT SAVED:", archive_path)
print("Independent Tone dataset loaded successfully.")

Saving the dataset (0/1 shards):   0%|          | 0/71 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/150 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/204 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/150 [00:00<?, ? examples/s]

Loading independent Tone dataset...


README.md:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

consumer-electronics-support.jsonl: reconstructing file:   0%|          |  0.00B /  197MB            

consumer-electronics-support.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/99930 [00:00<?, ? examples/s]


Dataset rows: 99930
Columns: ['generation', 'metadata', 'quality', 'record_id', 'record_index', 'run_id', 'scenario', 'schema_version', 'source_id', 'subdomain', 'turns']

First example:
{'generation': {'judge_model_id': 'ollama_chat/gpt-oss:20b', 'model_id': 'ollama_chat/gpt-oss:20b'}, 'metadata': {'category': 'technical', 'channel': 'chat', 'created_at': datetime.datetime(2025, 8, 3, 6, 31, 13), 'priority': 'normal', 'resolution_status': 'resolved', 'resolved_at': datetime.datetime(2025, 8, 4, 18, 46, 31)}, 'quality': {'coherence_score': 1.0, 'rubric_id': 'coherence-v2'}, 'record_id': '9bc1eebf-26d4-5704-a49c-af89cf3deb23', 'record_index': 0, 'run_id': '8c56469b-f094-4686-90dc-29715bab4228', 'scenario': 'Duplicate $5 shipping charge appears on the invoice for order ORD-4321.', 'schema_version': '1.0.0', 'source_id': 'consumer-electronics-support.md@8d67eaf56b0a', 'subdomain': 'billing-invoice-discrepancy', 'turns': [{'content': "Hi, I was looking at my latest invoice and noticed tha

In [41]:
import random
import json
import zipfile
from pathlib import Path
from datasets import Dataset

# Keep only conversations whose final turn is an agent response
candidates = []

for row in tone_independent:
    turns = row["turns"]

    if len(turns) < 2:
        continue

    if turns[-1]["role"] != "agent":
        continue

    if not any(t["role"] == "customer" for t in turns[:-1]):
        continue

    # Input = conversation before the final agent response
    conversation = "\n".join(
        f"{t['role'].capitalize()}: {t['content']}"
        for t in turns[:-1]
    )

    candidates.append({
        "id": row["record_id"],
        "subject": row["scenario"],
        "body": conversation,
        "answer": turns[-1]["content"]
    })

# Reproducible random sample
rng = random.Random(42)
rng.shuffle(candidates)

assert len(candidates) >= 150, "Not enough valid conversations."

tone_eval = Dataset.from_list(candidates[:150])

# Replace the previous Tone evaluation set
frozen_eval["tone"] = tone_eval

# Save the new frozen dataset
tone_path = Path("/kaggle/working/experiment_checkpoint/evaluation_datasets/tone")
tone_eval.save_to_disk(str(tone_path))

tone_eval.to_json(
    "/kaggle/working/experiment_checkpoint/tone_independent_eval.jsonl"
)

# Update metadata
metadata["evaluation_sources"]["tone"] = (
    "arpieb/consumer-electronics-support; "
    "150 sampled conversations; seed 42; "
    "input excludes final agent turn; reference is final agent turn"
)

metadata["tone_evaluation_count"] = len(tone_eval)

with open(
    "/kaggle/working/experiment_checkpoint/experiment_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

# Refresh checkpoint archive
root = Path("/kaggle/working/experiment_checkpoint")
archive_path = "/kaggle/working/phase1_base_checkpoint.zip"

with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in root.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(root.parent))

print("Independent Tone evaluation set frozen.")
print("Valid conversations found:", len(candidates))
print("Selected examples:", len(tone_eval))
print("Seed: 42")
print("Checkpoint archive updated.")
print("\nSample input:\n", tone_eval[0]["body"][:500])
print("\nReference reply:\n", tone_eval[0]["answer"][:500])

Saving the dataset (0/1 shards):   0%|          | 0/150 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Independent Tone evaluation set frozen.
Valid conversations found: 59589
Selected examples: 150
Seed: 42
Checkpoint archive updated.

Sample input:
 Customer: Hi, my Bluetooth speaker, the SoundWave X200, has stopped working after two weeks. I'd like to claim the warranty.
Agent: I'm sorry to hear that. Can you give me your order number so I can pull up the details?
Customer: Sure, it's ORD-3456.
Agent: Thanks. Let me check that for you. One moment... Okay, I see the order. The product is under warranty. I can arrange a replacement or repair. Which would you prefer?
Customer: I'd prefer a replacement. It's been frustrating.
Agent: Understood

Reference reply:
 You'll get the replacement within 5–7 business days after we receive the return. Please return the old speaker; it's not needed. I've sent the label and the instructions to your email. Let me know if you need anything else. Have a good day!


In [42]:
import torch
import gc
import json
from pathlib import Path
from tqdm.auto import tqdm

# Ensure only the original Base model is active
model.disable_adapter_layers()
model.eval()

output_path = Path(
    "/kaggle/working/results/base/tone_base.jsonl"
)
output_path.parent.mkdir(parents=True, exist_ok=True)

# Resume safely if this cell is interrupted
completed = {}

if output_path.exists():
    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            completed[r["example_id"]] = r

for i, example in enumerate(tqdm(tone_eval, desc="Base Tone")):

    example_id = str(example["id"])

    if example_id in completed:
        continue

    try:
        prediction = generate_response(
            "tone",
            example,
            max_new_tokens=250
        )

        result = {
            "example_id": example_id,
            "prediction": prediction,
            "reference": example["answer"]
        }

    except torch.cuda.OutOfMemoryError as e:
        gc.collect()
        torch.cuda.empty_cache()

        result = {
            "example_id": example_id,
            "prediction": "",
            "reference": example["answer"],
            "error": "CUDA OOM"
        }

    except Exception as e:
        result = {
            "example_id": example_id,
            "prediction": "",
            "reference": example["answer"],
            "error": str(e)
        }

    completed[example_id] = result

    # Save after every example
    with open(output_path, "w", encoding="utf-8") as f:
        for r in completed.values():
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

successful = sum(
    1 for r in completed.values()
    if "error" not in r and r["prediction"].strip()
)

print("\nBASE TONE EVALUATION")
print("Successful:", successful, "/", len(tone_eval))
print("Saved:", output_path)

Base Tone:   0%|          | 0/150 [00:00<?, ?it/s]


BASE TONE EVALUATION
Successful: 300 / 150
Saved: /kaggle/working/results/base/tone_base.jsonl


In [43]:
import json
import shutil
import zipfile
from pathlib import Path

# Paths
base_file = Path("/kaggle/working/results/base/tone_base.jsonl")
checkpoint = Path("/kaggle/working/experiment_checkpoint")
checkpoint_base = checkpoint / "base_results"
checkpoint_base.mkdir(parents=True, exist_ok=True)

# IDs belonging to the current independent Tone evaluation set
valid_ids = {str(example["id"]) for example in tone_eval}

# Read existing results and retain only current evaluation examples
records = {}
with open(base_file, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["example_id"] in valid_ids:
            records[r["example_id"]] = r

# Verify that all 150 examples are represented
print("Current evaluation examples:", len(valid_ids))
print("Matching result records:", len(records))
print("Successful predictions:", sum(
    1 for r in records.values()
    if "error" not in r and r["prediction"].strip()
))

assert len(records) == len(valid_ids), (
    "Some current Tone evaluation results are missing. "
    "Do not proceed until this is resolved."
)

# Rewrite the file with only the current 150 results
with open(base_file, "w", encoding="utf-8") as f:
    for r in records.values():
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Copy clean results into checkpoint
shutil.copy2(
    base_file,
    checkpoint_base / "tone_base.jsonl"
)

# Refresh checkpoint archive
archive_path = Path("/kaggle/working/phase1_base_checkpoint.zip")

with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in checkpoint.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(checkpoint.parent))

print("\nTone results cleaned and checkpoint updated.")
print("Archive:", archive_path)

Current evaluation examples: 150
Matching result records: 150
Successful predictions: 150

Tone results cleaned and checkpoint updated.
Archive: /kaggle/working/phase1_base_checkpoint.zip


In [44]:
import json
import zipfile
from pathlib import Path

base_dir = Path("/kaggle/working/results/base")
archive_path = Path("/kaggle/working/phase1_base_checkpoint.zip")

expected = {
    "json": 71,
    "tone": 150,
    "sql": 204,
    "samsum": 150
}

print("=" * 60)
print("PHASE 1 — BASELINE HANDOFF VERIFICATION")
print("=" * 60)

all_results_ready = True

for task, expected_count in expected.items():

    path = base_dir / f"{task}_base.jsonl"

    if not path.exists():
        print(f"\n{task.upper()}: RESULT FILE MISSING")
        all_results_ready = False
        continue

    with open(path, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]

    successful = sum(
        1 for r in records
        if "error" not in r and r.get("prediction", "").strip()
    )

    errors = len(records) - successful

    print(f"\n{task.upper()}")
    print("Expected examples:", expected_count)
    print("Saved records:", len(records))
    print("Successful:", successful)
    print("Errors:", errors)

    if len(records) != expected_count:
        all_results_ready = False

print("\n" + "=" * 60)

if archive_path.exists():
    with zipfile.ZipFile(archive_path, "r") as z:
        bad_file = z.testzip()
        print("Checkpoint ZIP:", "VALID" if bad_file is None else "CORRUPT")
        print("Files inside ZIP:", len(z.namelist()))
else:
    print("Checkpoint ZIP: MISSING")
    all_results_ready = False

print("\nBASELINE FILE VERIFICATION:",
      "READY" if all_results_ready else "CHECK REQUIRED")

print("Archive:", archive_path)
print("=" * 60)

PHASE 1 — BASELINE HANDOFF VERIFICATION

JSON
Expected examples: 71
Saved records: 71
Successful: 70
Errors: 1

TONE
Expected examples: 150
Saved records: 150
Successful: 150
Errors: 0

SQL
Expected examples: 204
Saved records: 204
Successful: 204
Errors: 0

SAMSUM
Expected examples: 150
Saved records: 150
Successful: 150
Errors: 0

Checkpoint ZIP: VALID
Files inside ZIP: 19

BASELINE FILE VERIFICATION: READY
Archive: /kaggle/working/phase1_base_checkpoint.zip


In [45]:

import torch
import gc
import json
import shutil
import zipfile
from pathlib import Path
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

MAX_NEW_TOKENS = {
    "json": 400,
    "tone": 250,
    "sql": 200,
    "samsum": 150
}

tasks = ["json", "tone", "sql", "samsum"]

solo_dir = Path("/kaggle/working/results/solo")
checkpoint_dir = Path("/kaggle/working/experiment_checkpoint")
checkpoint_solo_dir = checkpoint_dir / "solo_results"

solo_dir.mkdir(parents=True, exist_ok=True)
checkpoint_solo_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. CHECK ADAPTERS AND EVALUATION DATASETS
# --------------------------------------------------

print("=" * 60)
print("SOLO ADAPTER EVALUATION")
print("=" * 60)

print("Available adapters:", model.peft_config.keys())

for task in tasks:
    assert task in model.peft_config, f"Missing adapter: {task}"
    assert task in frozen_eval, f"Missing evaluation dataset: {task}"

    print(f"{task.upper()}: {len(frozen_eval[task])} examples")

# --------------------------------------------------
# 3. EVALUATE EACH ADAPTER ON ITS OWN TASK
# --------------------------------------------------

model.enable_adapter_layers()
model.eval()

solo_summary = {}

for task in tasks:

    print("\n" + "=" * 60)
    print(f"Evaluating SOLO adapter: {task.upper()}")
    print("=" * 60)

    # Activate only the adapter corresponding to this task
    model.set_adapter(task)
    model.eval()

    print("Active adapter:", model.active_adapters)

    output_path = solo_dir / f"{task}_solo.jsonl"

    # Resume previously completed predictions
    completed = {}

    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    completed[record["example_id"]] = record

    dataset = frozen_eval[task]

    for i, example in enumerate(
        tqdm(dataset, desc=f"Solo {task.upper()}")
    ):

        example_id = str(example.get("id", i))

        if example_id in completed:
            continue

        try:
            prediction = generate_response(
                task,
                example,
                max_new_tokens=MAX_NEW_TOKENS[task]
            )

            result = {
                "example_id": example_id,
                "prediction": prediction,
                "reference": get_reference(task, example)
            }

        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()

            result = {
                "example_id": example_id,
                "prediction": "",
                "reference": get_reference(task, example),
                "error": "CUDA OOM"
            }

        except Exception as e:

            result = {
                "example_id": example_id,
                "prediction": "",
                "reference": get_reference(task, example),
                "error": str(e)
            }

        completed[example_id] = result

        # Save after every example
        with open(output_path, "w", encoding="utf-8") as f:
            for record in completed.values():
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

    # Count results
    successful = sum(
        1 for r in completed.values()
        if "error" not in r and r["prediction"].strip()
    )

    errors = len(completed) - successful

    solo_summary[task] = {
        "expected": len(dataset),
        "saved": len(completed),
        "successful": successful,
        "errors": errors,
        "result_file": str(output_path)
    }

    print(f"\n{task.upper()} SUMMARY")
    print("Expected:", len(dataset))
    print("Saved:", len(completed))
    print("Successful:", successful)
    print("Errors:", errors)

    # Copy this task's result into the checkpoint
    shutil.copy2(
        output_path,
        checkpoint_solo_dir / f"{task}_solo.jsonl"
    )

    gc.collect()
    torch.cuda.empty_cache()

# --------------------------------------------------
# 4. SAVE SUMMARY AND UPDATE CHECKPOINT
# --------------------------------------------------

summary_path = solo_dir / "solo_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(solo_summary, f, indent=4)

shutil.copy2(
    summary_path,
    checkpoint_solo_dir / "solo_summary.json"
)

# Save a metadata snapshot
metadata_path = checkpoint_dir / "experiment_metadata.json"

if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
else:
    metadata = {}

metadata["solo_evaluation"] = solo_summary

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

# Rebuild checkpoint ZIP
archive_path = Path("/kaggle/working/phase1_base_checkpoint.zip")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for file in checkpoint_dir.rglob("*"):
        if file.is_file():
            z.write(
                file,
                file.relative_to(checkpoint_dir.parent)
            )

# --------------------------------------------------
# 5. FINAL REPORT
# --------------------------------------------------

print("\n" + "=" * 60)
print("SOLO EVALUATION — FINAL REPORT")
print("=" * 60)

total_expected = 0
total_successful = 0
total_errors = 0

for task, result in solo_summary.items():

    print(
        f"{task.upper():8} | "
        f"{result['successful']}/{result['expected']} successful | "
        f"{result['errors']} errors"
    )

    total_expected += result["expected"]
    total_successful += result["successful"]
    total_errors += result["errors"]

print("-" * 60)
print("TOTAL EXPECTED:", total_expected)
print("TOTAL SUCCESSFUL:", total_successful)
print("TOTAL ERRORS:", total_errors)

print("\nCheckpoint updated:", archive_path)
print("Summary saved:", summary_path)

print("=" * 60)


SOLO ADAPTER EVALUATION
Available adapters: dict_keys(['json', 'tone', 'sql', 'samsum'])
JSON: 71 examples
TONE: 150 examples
SQL: 204 examples
SAMSUM: 150 examples

Evaluating SOLO adapter: JSON
Active adapter: ['json']


Solo JSON:   0%|          | 0/71 [00:00<?, ?it/s]


JSON SUMMARY
Expected: 71
Saved: 71
Successful: 70
Errors: 1

Evaluating SOLO adapter: TONE
Active adapter: ['tone']


Solo TONE:   0%|          | 0/150 [00:00<?, ?it/s]


TONE SUMMARY
Expected: 150
Saved: 150
Successful: 150
Errors: 0

Evaluating SOLO adapter: SQL
Active adapter: ['sql']


Solo SQL:   0%|          | 0/204 [00:00<?, ?it/s]


SQL SUMMARY
Expected: 204
Saved: 204
Successful: 204
Errors: 0

Evaluating SOLO adapter: SAMSUM
Active adapter: ['samsum']


Solo SAMSUM:   0%|          | 0/150 [00:00<?, ?it/s]


SAMSUM SUMMARY
Expected: 150
Saved: 150
Successful: 150
Errors: 0

SOLO EVALUATION — FINAL REPORT
JSON     | 70/71 successful | 1 errors
TONE     | 150/150 successful | 0 errors
SQL      | 204/204 successful | 0 errors
SAMSUM   | 150/150 successful | 0 errors
------------------------------------------------------------
TOTAL EXPECTED: 575
TOTAL SUCCESSFUL: 574
TOTAL ERRORS: 1

Checkpoint updated: /kaggle/working/phase1_base_checkpoint.zip
Summary saved: /kaggle/working/results/solo/solo_summary.json
